## Fetching the content

### 1. Get all the article URLs

In [2]:
import requests
from bs4 import BeautifulSoup

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36"}
# Collect all the article urls here
article_urls = []

# Generate all the search page URLs
search_pages = (
    ['https://bioengineering.hyperbook.mcgill.ca/search-essays/'] + 
    [f'https://bioengineering.hyperbook.mcgill.ca/search-essays/page/{i}/' for i in range(2, 14)]
)

for search_page_url in search_pages:
    print(f'Scraping {search_page_url}')
    response = requests.get(search_page_url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    results = soup.find(id='results')
    
    for a in results.find_all('a'):
        if 'search-essays' not in a['href']:
            article_urls.append(a['href'])

print('Number of article URLs:', len(article_urls))

Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/2/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/3/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/4/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/5/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/6/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/7/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/8/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/9/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/10/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/11/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/12/
Scraping https://bioengineering.hyperbook.mcgill.ca/search-essays/page/13/
Number of article URLs: 255


### 2. Process each article

In [3]:
import requests
from bs4 import BeautifulSoup, Tag

class Article:
    # For any quirky naming of the references section
    ref_labels = ['references', 'bibliography']
    ref_headers = ['h2', 'h3']

    def __init__(self, url):
        self.url = url
        self.title = ""
        self.authors = ""
        self.abstract = ""
        self.content = ""
        self.references = []
    
    def extract_info(self):
        # Initialize the article soup
        response = requests.get(self.url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        article_soup = soup.find('article')

        self.title = self.extract_title(article_soup)
        self.authors = self.extract_authors(article_soup)
        self.abstract = self.extract_abstract(article_soup)
        self.content = self.extract_content(article_soup)
        self.references = self.extract_references(article_soup)

    def extract_title(self, article_soup):
        """
        Extracts the article's title. 
        Assumes that there is only one <h1>.
        """
        try:
            title = article_soup.find('h1').get_text()
            return title
        except Exception as e:
            print(f'Title extraction failed for {self.url}', e)
            return None

    def extract_authors(self, article_soup):
        """
        Extracts the article's authors.
        Assumes that they are found in the first <p>.
        """
        try:
            authors = article_soup.find('p').get_text()
            return authors
        except Exception as e:
            print(f'Author extraction failed for {self.url}', e)
            return None
        
    def extract_abstract(self, article_soup):
        """
        Extracts the article's abstract.
        Assumes that it's in a <p> following an <h2> containing 'abstract.'
        """
        try:
            abstract_header = None
            for child in article_soup.children:
                if isinstance(child, Tag):
                    # Check if it's a <p> tag and contains "references"
                    child_content = child.get_text(strip=True).lower()
                    if child.name in ['h2', 'h3'] and 'abstract' in child_content:
                        abstract_header = child
                        break

            abstract = abstract_header.find_next_sibling('p').get_text()
            return abstract
        
        except Exception as e:
            print(f'Abstract extraction failed for {self.url}', e)
            return None

    def extract_content(self, article_soup):
        """
        Extracts the article's content.
        Assumes that it ranges from the title up until the references section.
        """
        try:
            content = []    # Content that precedes references
            for child in article_soup.children:
                if isinstance(child, Tag):
                    # Check if it's a <p> tag and contains "references"
                    child_content = child.get_text(strip=True).lower()
                    if child.name in ['h2', 'h3'] and any(label in child_content for label in Article.ref_labels): 
                        break
                    content.append(child)

            # Create a new BeautifulSoup object to wrap the filtered content
            filtered_html = ''.join(str(tag) for tag in content)
            clean_soup = BeautifulSoup(filtered_html, 'html.parser')
            text_content = clean_soup.get_text(" ")

            return text_content
        
        except Exception as e:
            print(f'Content extraction failed for {self.url}', e)
            return None

    def extract_references(self, article_soup):
        """
        Extracts the article's references and concatenate them into a string.
        Assumes that they are contained in <p> following the <h2> references header.
        """
        try:
            references = []
            references_header = None

            for child in article_soup.children:
                if isinstance(child, Tag):
                    # Check if it's a header tag and contains "references"
                    child_content = child.get_text(strip=True).lower()
                    if child.name in ['h2', 'h3'] and any(label in child_content for label in Article.ref_labels):
                        references_header = child
                        break

            for sibling in references_header.find_next_siblings():
                if sibling.name == 'p':
                    reference = sibling.get_text().replace("\n", " ")
                    references.append(reference)

                elif sibling.name == 'ol':
                    for ol_child in sibling.children:
                        if isinstance(ol_child, Tag):
                            content = ol_child.get_text().replace("\n", "")
                            references.append(content)
                else:
                    break
                
            if len(references) == 0:
                print(f'Empty references for {self.url}')
                return None

            return "\n".join(references)
        
        except Exception as e:
            print(f'References extraction failed for {self.url}', e)
            return None

In [4]:
from tqdm import tqdm
articles_data = []

for index, url in enumerate(tqdm(article_urls)):
    article = Article(url=url)
    article.extract_info()
    articles_data.append(vars(article))

 81%|████████  | 206/255 [10:41<01:59,  2.44s/it]

Abstract extraction failed for https://bioengineering.hyperbook.mcgill.ca/bioinformatics-and-sensory-function-of-mammalian-fur-an-analysis-of-coat-patterning-whiskers-and-trigger-hairs-hair-fur-3/ 'NoneType' object has no attribute 'find_next_sibling'


 81%|████████  | 207/255 [10:43<01:47,  2.23s/it]

Abstract extraction failed for https://bioengineering.hyperbook.mcgill.ca/sleeping-in-flight-adaptations-of-a-birds-brain-wing-3/ 'NoneType' object has no attribute 'find_next_sibling'


 82%|████████▏ | 210/255 [10:49<01:42,  2.27s/it]

Abstract extraction failed for https://bioengineering.hyperbook.mcgill.ca/cephalopods-and-their-beak-beak-3/ 'NoneType' object has no attribute 'find_next_sibling'


... (31 outputs left out)

Abstract extraction failed for https://bioengineering.hyperbook.mcgill.ca/exploring-the-structure-and-functionality-of-flippers-flipper-1/ 'NoneType' object has no attribute 'find_next_sibling'


 98%|█████████▊| 249/255 [12:36<00:17,  2.87s/it]

Abstract extraction failed for https://bioengineering.hyperbook.mcgill.ca/1292/ 'NoneType' object has no attribute 'find_next_sibling'


100%|██████████| 255/255 [12:47<00:00,  3.01s/it]


In [6]:
import pandas as pd

df = pd.DataFrame(articles_data)
df.to_csv('../../data/raw/hyperbook.csv', index=False)
df.head()

,url,title,authors,abstract,content,references
0,https://bioengineering.hyperbook.mcgill.ca/soc...,Social Wasps: A Chemical Analysis,"Andrew Habelrih, Eliot Azar, Malak Zakaria",This paper discusses the different chemical an...,Social Wasps: A Chemical Analysis Andrew Habe...,"Alhakamy, N. A., Ahmed, O. A., Md, S., & Fahmy..."
1,https://bioengineering.hyperbook.mcgill.ca/tin...,Tiny Alchemists: The Chemical Foundations of T...,"Rosalie Beaudin, Naia Kim, Mattias Sucher, Luc...","Despite their individual simplicity, termites ...",Tiny Alchemists: The Chemical Foundations of T...,"Ana Maria Costa-Leonardo, I. H. (2010). Chapte..."
2,https://bioengineering.hyperbook.mcgill.ca/mar...,Marvels of Chemistry: Murmurations and their C...,"Maeve Ding, Matt Levasseur, Nicole Shen, Racha...",This paper investigates the many chemical proc...,Marvels of Chemistry: Murmurations and their C...,"Awerman, J. L., & Romero, L. M. (2010). Chroni..."
3,https://bioengineering.hyperbook.mcgill.ca/phy...,Physarum Polycephalum (Slime Mold): Chemical P...,"Olivia Cozariuc, Tamir Kegeles, Ethan Levene, ...",The protist true slime mold Physarum polycepha...,Physarum Polycephalum (Slime Mold): Chemical P...,"Achenbach, F., Naib-Majani, W., & Wohlfarth-Bo..."
4,https://bioengineering.hyperbook.mcgill.ca/a-c...,A Chemical Composition to Make Them Cower: A D...,"Olivia Ghiaur, Camille Heaney, Chris Hu, Vadim...",Sponges have several feeding modes and use a v...,A Chemical Composition to Make Them Cower: A D...,"Agarwal, V., Blanton, J. M., Podell, S., Taton..."
